In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

print(module_path)

import numpy as np
import torch
import torch.nn as nn
from hedging.envs import HedgeCallBS
from hedging.plot_utils import plot_portfolio_vs_option_price
from hedging.logit_normal import LogitNormal

from torchrl.envs import GymWrapper
from torchrl.envs.utils import ExplorationType, set_exploration_type
from torchrl.modules import SafeProbabilisticModule

import torch.nn as nn
from tensordict.nn import TensorDictModule, TensorDictSequential


/Users/manu13/Desktop/PHD/DeepHedging/deep_hedging_v0


In [ ]:
# --- Env Parameters ---
S0 = np.array([50.0, 100.0, 200.0])
K  = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
maturity = 1.0
r = 0.03
sigma = np.array([0.15, 0.2, 0.25])
num_paths = 100
num_steps = 250
history_len = 1
input_dim = 11
hidden_size = 64
action_dim = 1

base_env = HedgeCallBS(S0, K, maturity, r, sigma, num_paths, num_steps, history_len=history_len)
env = GymWrapper(base_env)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
act_spec = env.specs["input_spec", "full_action_spec", "action"].to(device)

In [ ]:
class LatestState(nn.Module):
    def forward(self, obs):
        # obs is typically [B, 1, D] when history_len=1, but may be [B, D] on some wrappers
        if obs.dim() == 3 and obs.size(-2) == 1:
            return obs[..., -1, :]  # [B, D]
        elif obs.dim() == 2:
            return obs  # already [B, D]
        else:
            
            return obs[..., -1, :]

latest_state = TensorDictModule(
    LatestState(),
    in_keys=["observation"],
    out_keys=["features"],
)

# --- Policy head: tanh + Softplus with min std (stability) ---
class PolicyHead(nn.Module):
    def __init__(self, input_dim, hidden_size, action_dim=1, min_std=1e-3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_size)
        self.fc_mu = nn.Linear(hidden_size, action_dim)
        self.fc_sigma = nn.Linear(hidden_size, action_dim)
        self.softplus = nn.Softplus()
        self.min_std = min_std
        nn.init.zeros_(self.fc_mu.bias)
        nn.init.zeros_(self.fc_sigma.bias)

    def forward(self, x):
        h = torch.tanh(self.fc1(x))
        loc = self.fc_mu(h)
        scale = self.softplus(self.fc_sigma(h))
        scale = torch.clamp(scale, min=self.min_std)
        return loc, scale

policy_head = TensorDictModule(
    PolicyHead(input_dim=input_dim, hidden_size=hidden_size, action_dim=action_dim),
    in_keys=["features"],
    out_keys=["loc", "scale"],  # IMPORTANT: match your LogitNormal signature
)

actor = SafeProbabilisticModule(
    in_keys=["loc", "scale"],
    out_keys=["action"],
    distribution_class=LogitNormal,      # same as baseline (samples in (0,1))
    spec=act_spec,                       # spec is [0,1], aligns with LogitNormal support
    return_log_prob=True,
    log_prob_key="action_log_prob",
)

actor = TensorDictSequential(latest_state, policy_head, actor).to(device)


In [5]:
num_epochs = 20
num_episodes = 200
gamma = 0.999
learning_rate = 1e-5
optimizer = torch.optim.Adam(actor.parameters(), lr=learning_rate)

In [6]:
def discounted_returns(rewards: torch.Tensor, gamma: float) -> torch.Tensor:
    # rewards: [T, B]
    T, B = rewards.shape
    idx = torch.arange(T, device=rewards.device)
    exp_mat = idx[None, :] - idx[:, None]             # [T, T]
    disc_mat = torch.triu(torch.pow(
        torch.as_tensor(gamma, device=rewards.device), exp_mat).to(rewards.dtype)
    )
    return disc_mat @ rewards                          # [T, B]

In [7]:
with set_exploration_type(ExplorationType.RANDOM):
    for epoch in range(num_epochs):
        for episode in range(num_episodes):
            # mirror baseline seeding cadence
            env.set_seed(epoch + 1000)

            actor.train()
            td = env.rollout(
                policy=actor,
                max_steps=int(num_steps * maturity),
                auto_reset=True,
                auto_cast_to_device=True,
                break_when_all_done=True,   # full ep for all envs
            )

            # shapes: [T, B]
            rewards = td["next", "reward"].squeeze(-1)
            log_probs = td["action_log_prob"].squeeze(-1)

            # discounted returns and normalization across envs (axis=1)
            R = discounted_returns(rewards, gamma)     # [T, B]
            eps = torch.finfo(R.dtype).eps
            R = R - R.mean(dim=1, keepdim=True)
            R = R / (R.std(dim=1, keepdim=True) + eps)

            # reinforce loss
            loss = (-(R * log_probs)).mean()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(actor.parameters(), max_norm=1.0)
            # nansafe (shouldn’t trigger, but keeps parity with your PPO hygiene)
            for p in actor.parameters():
                if p.grad is not None:
                    p.grad = torch.nan_to_num(p.grad)
            optimizer.step()

            if (episode + 1) % 10 == 0:
                avg_reward = rewards.mean().item()     # per-step mean over T×B
                print(
                    f"Epoch {epoch+1}/{num_epochs}, Episode {episode + 1}/{num_episodes}, "
                    f"Loss: {loss.item():.6f}, Avg. Reward: {avg_reward:.6f}"
                )

Epoch 1/20, Episode 10/200, Loss: -0.011372, Avg. Reward: -4.698537
Epoch 1/20, Episode 20/200, Loss: -0.011698, Avg. Reward: -4.506646
Epoch 1/20, Episode 30/200, Loss: -0.009935, Avg. Reward: -4.615551
Epoch 1/20, Episode 40/200, Loss: -0.012358, Avg. Reward: -4.562247
Epoch 1/20, Episode 50/200, Loss: -0.012689, Avg. Reward: -4.696561
Epoch 1/20, Episode 60/200, Loss: -0.013144, Avg. Reward: -4.423946
Epoch 1/20, Episode 70/200, Loss: -0.014809, Avg. Reward: -4.498731
Epoch 1/20, Episode 80/200, Loss: -0.011040, Avg. Reward: -4.431716
Epoch 1/20, Episode 90/200, Loss: -0.012917, Avg. Reward: -4.641775
Epoch 1/20, Episode 100/200, Loss: -0.013966, Avg. Reward: -4.504641
Epoch 1/20, Episode 110/200, Loss: -0.016617, Avg. Reward: -4.461145
Epoch 1/20, Episode 120/200, Loss: -0.010672, Avg. Reward: -4.582721
Epoch 1/20, Episode 130/200, Loss: -0.014548, Avg. Reward: -4.778581
Epoch 1/20, Episode 140/200, Loss: -0.013379, Avg. Reward: -4.543631
Epoch 1/20, Episode 150/200, Loss: -0.01491

In [11]:
base_env = HedgeCallBS(S0, K, maturity, r, sigma, 5, num_steps, history_len=history_len)
env = GymWrapper(base_env, device=device)
env.reset(seed=0)

with set_exploration_type(ExplorationType.DETERMINISTIC):
    rollout = env.rollout(max_steps=num_steps, policy=actor)


In [12]:
rewards = rollout['next', 'reward'].detach().cpu().numpy()
rewards.min(), rewards.max(), rewards.mean(), rewards.std()

(-38.29965, -9.219492e-05, -3.8068147, 5.236656)

In [13]:
plot_portfolio_vs_option_price(env._env)